# 네이버 검색 트렌드 시계열 차트

다이소 뷰티 engagement_score TOP50 제품의 검색 트렌드를 시각화한다.  
데이터: `06_analysis/04_search_trend/output/search_trend_product_base_20260225.csv`  
차트 저장: `04_outputs/figures/search_trend/`

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import numpy as np

# 한글 폰트
plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False

# 인라인 출력
%matplotlib inline

In [ ]:
# ── 경로 설정 ──────────────────────────────────────────
PROJECT_ROOT = Path.cwd().parent.parent  # Why-pi/
OUTPUT_DIR = PROJECT_ROOT / "06_analysis" / "04_search_trend" / "output"
CHART_DIR = PROJECT_ROOT / "04_outputs" / "figures" / "search_trend"
CHART_DIR.mkdir(parents=True, exist_ok=True)

# 색상 팔레트
COLORS = [
    "#E74C3C", "#3498DB", "#2ECC71", "#F39C12", "#9B59B6",
    "#1ABC9C", "#E67E22", "#34495E", "#E91E63", "#00BCD4",
]

In [ ]:
# ── 데이터 로드 ────────────────────────────────────────
base_df = pd.read_csv(OUTPUT_DIR / "search_trend_product_base_20260225.csv")
base_df["period"] = pd.to_datetime(base_df["period"])

anchor_df = pd.read_csv(OUTPUT_DIR / "search_trend_product_anchor_20260225.csv")
anchor_df["period"] = pd.to_datetime(anchor_df["period"])

finals_df = pd.read_csv(OUTPUT_DIR / "search_trend_product_finals_20260225.csv")
finals_df["period"] = pd.to_datetime(finals_df["period"])

segment_df = pd.read_csv(OUTPUT_DIR / "search_trend_product_segment_20260225.csv")
segment_df["period"] = pd.to_datetime(segment_df["period"])

mapping_df = pd.read_csv(OUTPUT_DIR / "keyword_mapping_20260225.csv")

# 앵커 정규화 전체(base) 서브셋
base_anchor = anchor_df[anchor_df["segment_type"].isna()].copy()

# 앵커 정규화 순위
anchor_ranking = (
    base_anchor.groupby("keyword_group")["ratio_normalized"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
anchor_ranking.columns = ["제품", "정규화 점수"]
anchor_ranking.index = range(1, len(anchor_ranking) + 1)
anchor_ranking.index.name = "순위"
anchor_ranking["정규화 점수"] = anchor_ranking["정규화 점수"].round(1)

print(f"행 수: {len(base_df):,}")
print(f"기간: {base_df['period'].min().strftime('%Y-%m')} ~ {base_df['period'].max().strftime('%Y-%m')}")
print(f"제품 수: {base_df['keyword_group'].nunique()}")
base_df.head()

## 유틸리티 함수

In [ ]:
def plot_products(
    df: pd.DataFrame,
    products: list[str],
    title: str,
    filename: str = None,
    figsize: tuple = (14, 6),
    highlight: str = None,
    annotate_peaks: bool = False,
    ylabel: str = "검색 트렌드 (ratio)",
    value_col: str = "ratio",
):
    """제품별 시계열 차트 생성"""
    fig, ax = plt.subplots(figsize=figsize)

    for i, product in enumerate(products):
        sub = df[df["keyword_group"] == product].sort_values("period")
        if sub.empty:
            continue

        color = COLORS[i % len(COLORS)]
        lw = 2.5 if product == highlight else 1.8
        alpha = 1.0 if product == highlight else 0.8
        zorder = 10 if product == highlight else 5

        ax.plot(
            sub["period"], sub[value_col],
            label=product, color=color,
            linewidth=lw, alpha=alpha, zorder=zorder,
            marker="o", markersize=3,
        )

        if annotate_peaks:
            peak_idx = sub[value_col].idxmax()
            peak_row = sub.loc[peak_idx]
            ax.annotate(
                f'{peak_row[value_col]:.0f}',
                xy=(peak_row["period"], peak_row[value_col]),
                xytext=(0, 8), textcoords="offset points",
                fontsize=8, ha="center", color=color, fontweight="bold",
            )

    ax.set_title(title, fontsize=14, fontweight="bold", pad=12)
    ax.set_xlabel("")
    ax.set_ylabel(ylabel, fontsize=11)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    plt.xticks(rotation=45, ha="right")
    ax.legend(loc="upper right", fontsize=9, framealpha=0.9)
    ax.grid(True, alpha=0.3, linestyle="--")
    ax.set_xlim(
        df["period"].min() - pd.Timedelta(days=15),
        df["period"].max() + pd.Timedelta(days=15),
    )
    plt.tight_layout()

    if filename:
        path = CHART_DIR / filename
        fig.savefig(path, dpi=150, bbox_inches="tight", facecolor="white")
        print(f"  저장: {path.name}")

    plt.show()


def plot_heatmap(
    df: pd.DataFrame,
    products: list[str],
    title: str,
    filename: str = None,
    value_col: str = "ratio",
):
    """제품×월 히트맵"""
    existing = [p for p in products if p in df["keyword_group"].unique()]
    sub = df[df["keyword_group"].isin(existing)].copy()
    pivot = sub.pivot_table(
        index="keyword_group", columns="period", values=value_col, aggfunc="first"
    )
    pivot = pivot.reindex(existing)

    fig, ax = plt.subplots(figsize=(16, max(4, len(existing) * 0.45)))
    im = ax.imshow(pivot.values, aspect="auto", cmap="YlOrRd", interpolation="nearest")

    ax.set_yticks(range(len(existing)))
    ax.set_yticklabels(existing, fontsize=9)

    periods = [p.strftime("%Y-%m") for p in pivot.columns]
    ax.set_xticks(range(len(periods)))
    ax.set_xticklabels(periods, rotation=45, ha="right", fontsize=8)

    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            val = pivot.values[i, j]
            if not np.isnan(val):
                color = "white" if val > 50 else "black"
                ax.text(j, i, f"{val:.0f}", ha="center", va="center", fontsize=6, color=color)

    plt.colorbar(im, ax=ax, shrink=0.8, label=value_col)
    ax.set_title(title, fontsize=14, fontweight="bold", pad=12)
    plt.tight_layout()

    if filename:
        path = CHART_DIR / filename
        fig.savefig(path, dpi=150, bbox_inches="tight", facecolor="white")
        print(f"  저장: {path.name}")

    plt.show()


def plot_segment_comparison(
    segment_df: pd.DataFrame,
    products: list[str],
    seg_type: str,
    title: str,
    filename: str = None,
    figsize: tuple = (16, 5),
):
    """세그먼트별 평균 ratio 비교 바 차트"""
    sub = segment_df[
        (segment_df["segment_type"] == seg_type) &
        (segment_df["keyword_group"].isin(products))
    ].copy()

    if sub.empty:
        print(f"  [경고] {seg_type} 데이터 없음")
        return

    pivot = sub.pivot_table(
        index="keyword_group", columns="segment_label",
        values="ratio", aggfunc="mean"
    ).reindex(products).dropna(how="all")

    fig, ax = plt.subplots(figsize=figsize)
    pivot.plot(kind="bar", ax=ax, color=COLORS[:pivot.shape[1]], edgecolor="white", width=0.8)

    ax.set_title(title, fontsize=14, fontweight="bold", pad=12)
    ax.set_ylabel("평균 ratio", fontsize=11)
    ax.set_xlabel("")
    plt.xticks(rotation=45, ha="right", fontsize=9)
    ax.legend(title=seg_type, fontsize=9, framealpha=0.9)
    ax.grid(True, alpha=0.3, linestyle="--", axis="y")
    plt.tight_layout()

    if filename:
        path = CHART_DIR / filename
        fig.savefig(path, dpi=150, bbox_inches="tight", facecolor="white")
        print(f"  저장: {path.name}")

    plt.show()

---

## 0. 제품 수 추이

TOP 50 제품 중 검색 트렌드에 잡히는 제품 수의 월별 변화.  
2024년 초에는 소수 제품만 검색되었으나, 2025년 하반기에 크게 증가하여 시장 확대를 보여준다.

In [ ]:
# 기간별 제품 수 추이
counts = base_df.groupby("period")["keyword_group"].nunique().reset_index()
counts.columns = ["period", "product_count"]

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(counts["period"], counts["product_count"], width=25, color="#3498DB", alpha=0.7, edgecolor="#2980B9")
ax.plot(counts["period"], counts["product_count"], color="#E74C3C", linewidth=2, marker="o", markersize=5)

for _, row in counts.iterrows():
    ax.annotate(
        str(int(row["product_count"])),
        xy=(row["period"], row["product_count"]),
        xytext=(0, 8), textcoords="offset points",
        fontsize=8, ha="center", fontweight="bold",
    )

ax.set_title("기간별 검색 트렌드 제품 수 추이 (TOP 50)", fontsize=14, fontweight="bold", pad=12)
ax.set_ylabel("제품 수", fontsize=11)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45, ha="right")
ax.grid(True, alpha=0.3, linestyle="--", axis="y")
plt.tight_layout()

fig.savefig(CHART_DIR / "00_product_count_timeline.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show()

---

## 1. 전체 TOP 5 비교

In [ ]:
avg_ratio = base_df.groupby("keyword_group")["ratio"].mean().sort_values(ascending=False)
top5_products = avg_ratio.head(5).index.tolist()

print("기본 트렌드 평균 ratio TOP 5:")
for i, (name, val) in enumerate(avg_ratio.head(5).items(), 1):
    print(f"  {i}위: {name} ({val:.2f})")

plot_products(
    base_df, top5_products,
    "전체 TOP 5 제품 검색 트렌드",
    filename="02_top5_overall.png",
    annotate_peaks=True,
    highlight=top5_products[0],
)

---

## 2. VT 리들샷 시리즈 비교

VT 리들샷 100/300/500은 동일 라인의 강도별 제품이다.  
검색 트렌드에서 어떤 차이를 보이는지 확인한다.

In [ ]:
ridleshot_products = [p for p in base_df["keyword_group"].unique() if "리들샷" in p]
print(f"리들샷 시리즈: {ridleshot_products}")

plot_products(
    base_df, ridleshot_products,
    "VT 리들샷 시리즈 검색 트렌드 비교",
    filename="03_vt_ridleshot_series.png",
    highlight="VT 리들샷 100",
    annotate_peaks=True,
)

---

## 3. VT 전 제품 비교

VT 브랜드의 모든 TOP 50 진입 제품을 비교한다.

In [ ]:
vt_products = [p for p in base_df["keyword_group"].unique() if p.startswith("VT ")]
print(f"VT 제품 ({len(vt_products)}개): {vt_products}")

plot_products(
    base_df, vt_products,
    "VT 브랜드 전 제품 검색 트렌드",
    filename="04_vt_all_products.png",
    highlight="VT 리들샷 100",
    annotate_peaks=True,
)

---

## 4. 본셉 시리즈 비교

본셉 레티놀/비타씨 라인별 검색 트렌드를 비교한다.

In [ ]:
boncep_products = [p for p in base_df["keyword_group"].unique() if p.startswith("본셉")]
print(f"본셉 제품 ({len(boncep_products)}개): {boncep_products}")

plot_products(
    base_df, boncep_products,
    "본셉 시리즈 검색 트렌드 비교",
    filename="05_boncep_series.png",
    annotate_peaks=True,
)

---

## 5. 다이소 자체 브랜드 제품

다이소 브랜드명으로 판매되는 자체 제품의 검색 트렌드.

In [ ]:
daiso_products = [p for p in base_df["keyword_group"].unique() if p.startswith("다이소")]
print(f"다이소 자체 브랜드 제품 ({len(daiso_products)}개): {daiso_products}")

if daiso_products:
    plot_products(
        base_df, daiso_products,
        "다이소 자체 브랜드 제품 검색 트렌드",
        filename="06_daiso_own_brand.png",
        annotate_peaks=True,
    )
else:
    print("  다이소 자체 브랜드 제품 없음")

---

## 6. 앵커 정규화 TOP 20

engagement 1위(VT 리들샷 100)=100 기준으로 전체 제품을 동일 기준 정규화.

In [ ]:
top20 = anchor_ranking.head(20)

fig, ax = plt.subplots(figsize=(14, 8))
colors_bar = ["#E74C3C" if i == 0 else "#3498DB" for i in range(len(top20))]
bars = ax.barh(range(len(top20) - 1, -1, -1), top20["정규화 점수"], color=colors_bar, edgecolor="white")

ax.set_yticks(range(len(top20) - 1, -1, -1))
ax.set_yticklabels(top20["제품"], fontsize=10)

for i, (_, row) in enumerate(top20.iterrows()):
    ax.text(row["정규화 점수"] + 1, len(top20) - 1 - i, f"{row['정규화 점수']:.1f}",
            va="center", fontsize=9, fontweight="bold")

ax.set_title("앵커 정규화 TOP 20 (VT 리들샷 100 = 100)", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("정규화 점수", fontsize=11)
ax.axvline(x=100, color="#E74C3C", linestyle="--", alpha=0.5, label="앵커 기준 (100)")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, linestyle="--", axis="x")
plt.tight_layout()

fig.savefig(CHART_DIR / "01_anchor_top20_bar.png", dpi=150, bbox_inches="tight", facecolor="white")
print("  저장: 01_anchor_top20_bar.png")
plt.show()

---

## 7. 앵커 정규화 시계열 — TOP 10

앵커(VT 리들샷 100)=100 기준으로 정규화된 시계열.  
배치 간 직접 비교가 불가능한 raw ratio와 달리, 정규화 점수는 전 제품을 동일 기준으로 비교 가능.

In [ ]:
top10_anchor = anchor_ranking.head(10)["제품"].tolist()

plot_products(
    base_anchor, top10_anchor,
    "앵커 정규화 시계열 TOP 10 (VT 리들샷 100 = 100)",
    filename="07_anchor_top10_timeseries.png",
    highlight="VT 리들샷 100",
    value_col="ratio_normalized",
    ylabel="정규화 점수 (앵커=100)",
    annotate_peaks=True,
)

---

## 8. TOP 15 히트맵 (월별 검색량)

| 패턴 | 제품 | 히트맵 특징 |
|------|------|----------|
| 장기 강세형 | VT 리들샷 100 | 전 기간 진한 색 유지 |
| 추격자형 | VT 리들샷 300 | 100을 따라가되 50~60% 수준 |
| 니치 안정형 | 본셉 비타씨 크림, 본셉 레티놀 2500 | 중간 톤 꾸준히 유지 |
| 최근 등장형 | 에이솔루션 어성초, 다이소 에그캡슐팩 | 우측에 색 집중 |
| 하락형 | 손앤박 아티 스프레드 | 좌측 진하고 우측 연해짐 |

In [ ]:
top15_list = anchor_ranking.head(15)["제품"].tolist()

plot_heatmap(
    base_df, top15_list,
    "TOP 15 제품 월별 검색량 히트맵",
    filename="08_top15_heatmap.png",
)

---

## 9. 성별 비교 — TOP 10 제품

In [ ]:
# 성별 앵커 정규화 비교
for gender in ["남성", "여성"]:
    sub = anchor_df[
        (anchor_df["segment_type"] == "gender") &
        (anchor_df["segment_label"] == gender)
    ]
    if sub.empty:
        continue
    ranking = (
        sub.groupby("keyword_group")["ratio_normalized"]
        .mean()
        .sort_values(ascending=False)
        .head(10)
        .reset_index()
    )
    ranking.columns = ["제품", "정규화 점수"]
    ranking.index = range(1, len(ranking) + 1)
    ranking.index.name = "순위"
    ranking["정규화 점수"] = ranking["정규화 점수"].round(1)
    print(f"\n=== {gender} 앵커 정규화 TOP 10 ===")
    display(ranking)

In [ ]:
plot_segment_comparison(
    segment_df, top10_anchor, "gender",
    "성별 비교 — 앵커 정규화 TOP 10 제품",
    filename="09_gender_comparison.png",
)

---

## 10. 연령대별 비교 — TOP 5 제품

In [ ]:
for age in ["10대", "20대", "30대", "40대", "50대", "60대"]:
    sub = anchor_df[
        (anchor_df["segment_type"] == "age") &
        (anchor_df["segment_label"] == age)
    ]
    if sub.empty:
        continue
    ranking = (
        sub.groupby("keyword_group")["ratio_normalized"]
        .mean()
        .sort_values(ascending=False)
        .head(5)
        .reset_index()
    )
    ranking.columns = ["제품", "정규화 점수"]
    ranking.index = range(1, len(ranking) + 1)
    ranking.index.name = "순위"
    ranking["정규화 점수"] = ranking["정규화 점수"].round(1)
    print(f"\n=== {age} 앵커 정규화 TOP 5 ===")
    display(ranking)

In [ ]:
top5_anchor = anchor_ranking.head(5)["제품"].tolist()

plot_segment_comparison(
    segment_df, top5_anchor, "age",
    "연령대별 비교 — 앵커 정규화 TOP 5 제품",
    filename="10_age_comparison.png",
)

---

## 11. 기기별 비교 — TOP 10 제품

In [ ]:
plot_segment_comparison(
    segment_df, top10_anchor, "device",
    "기기별 비교 — 앵커 정규화 TOP 10 제품",
    filename="11_device_comparison.png",
)

---

## 12. 제품 수명주기 분류

| 수명주기 | 특징 | 대표 제품 | 재고 전략 |
|---------|------|----------|----------|
| 바이럴형 | 고평균+고변동 | VT 리들샷 100, 에이솔루션 어성초 | 피크 시기 집중 물량 |
| 안정형 | 고평균+저변동 | 본셉 비타씨, 본셉 레티놀 | Hub 매장 상시 재고 |
| 단발성 | 저평균+고변동 | VT PDRN 광채크림 | 시즌별 소량 운영 |
| 저관심 | 저평균+저변동 | 팩미인, 성분에디터 | 재고 최소화 |

In [ ]:
# ── 제품별 요약 통계 ────────────────────────────────────
product_summary = (
    base_df.groupby("keyword_group")["ratio"]
    .agg(["mean", "max", "min", "std", "count"])
    .round(2)
    .sort_values("mean", ascending=False)
)
product_summary.columns = ["평균", "최대", "최소", "표준편차", "데이터수"]

# 피크 시기 추출
peak_periods = (
    base_df.loc[base_df.groupby("keyword_group")["ratio"].idxmax()]
    [["keyword_group", "period"]]
    .set_index("keyword_group")
)
peak_periods.columns = ["피크시기"]
peak_periods["피크시기"] = peak_periods["피크시기"].dt.strftime("%Y-%m")

# 최근 ratio (2026-01)
latest = base_df[base_df["period"] == base_df["period"].max()].set_index("keyword_group")[["ratio"]]
latest.columns = ["최근(2026-01)"]

# 첫 등장 시기
first_appear = (
    base_df[base_df["ratio"] > 0]
    .groupby("keyword_group")["period"]
    .min()
    .to_frame("첫등장")
)
first_appear["첫등장"] = first_appear["첫등장"].dt.strftime("%Y-%m")

summary_full = product_summary.join(peak_periods).join(latest).join(first_appear)
print("=== 전체 제품 요약 통계 (평균 ratio 순) ===")
display(summary_full)

In [ ]:
# ── 제품 수명주기 분류 시각화 (평균 vs 표준편차) ────────
fig, ax = plt.subplots(figsize=(14, 8))

x = product_summary["평균"]
y = product_summary["표준편차"]
names = product_summary.index

ax.scatter(x, y, s=80, c="#3498DB", alpha=0.7, edgecolors="white", linewidth=0.5)

# 상위 15개 제품만 라벨 표시
for name in names[:15]:
    row = product_summary.loc[name]
    ax.annotate(
        name, xy=(row["평균"], row["표준편차"]),
        xytext=(5, 5), textcoords="offset points",
        fontsize=8, alpha=0.8,
    )

# 사분면 구분선
mean_x = x.median()
mean_y = y.median()
ax.axvline(x=mean_x, color="gray", linestyle="--", alpha=0.3)
ax.axhline(y=mean_y, color="gray", linestyle="--", alpha=0.3)

# 사분면 라벨
ax.text(x.max() * 0.7, y.max() * 0.9, "바이럴형\n(고평균+고변동)", fontsize=10, color="#E74C3C", ha="center")
ax.text(x.max() * 0.7, y.min() + 1, "안정형\n(고평균+저변동)", fontsize=10, color="#2ECC71", ha="center")
ax.text(x.min() + 1, y.max() * 0.9, "단발성\n(저평균+고변동)", fontsize=10, color="#F39C12", ha="center")
ax.text(x.min() + 1, y.min() + 1, "저관심\n(저평균+저변동)", fontsize=10, color="#95A5A6", ha="center")

ax.set_title("제품 수명주기 분류 (평균 ratio vs 표준편차)", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("평균 ratio", fontsize=11)
ax.set_ylabel("표준편차", fontsize=11)
ax.grid(True, alpha=0.3, linestyle="--")
plt.tight_layout()

fig.savefig(CHART_DIR / "12_lifecycle_scatter.png", dpi=150, bbox_inches="tight", facecolor="white")
print("  저장: 12_lifecycle_scatter.png")
plt.show()

---

## 13. 브랜드별 제품 포트폴리오 비교

In [ ]:
brand_counts = mapping_df["brand_name"].value_counts().reset_index()
brand_counts.columns = ["브랜드", "제품수"]

fig, ax = plt.subplots(figsize=(14, 6))
top_brands = brand_counts.head(15)
bars = ax.bar(range(len(top_brands)), top_brands["제품수"], color=COLORS[:len(top_brands)] * 3, edgecolor="white")

ax.set_xticks(range(len(top_brands)))
ax.set_xticklabels(top_brands["브랜드"], rotation=45, ha="right", fontsize=10)

for i, (_, row) in enumerate(top_brands.iterrows()):
    ax.text(i, row["제품수"] + 0.1, str(row["제품수"]),
            ha="center", fontsize=10, fontweight="bold")

ax.set_title("브랜드별 TOP 50 진입 제품 수", fontsize=14, fontweight="bold", pad=12)
ax.set_ylabel("제품 수", fontsize=11)
ax.grid(True, alpha=0.3, linestyle="--", axis="y")
plt.tight_layout()

fig.savefig(CHART_DIR / "13_brand_portfolio.png", dpi=150, bbox_inches="tight", facecolor="white")
print("  저장: 13_brand_portfolio.png")
plt.show()

---

## 14. 세그먼트별 앵커 정규화 히트맵

In [ ]:
seg_labels = ["남성", "여성", "10대", "20대", "30대", "40대", "50대", "60대", "PC", "모바일"]
top10_names = anchor_ranking.head(10)["제품"].tolist()

heatmap_data = []
for seg_label in seg_labels:
    sub = anchor_df[
        (anchor_df["segment_label"] == seg_label) &
        (anchor_df["keyword_group"].isin(top10_names))
    ]
    avg = sub.groupby("keyword_group")["ratio_normalized"].mean()
    for product in top10_names:
        heatmap_data.append({
            "세그먼트": seg_label,
            "제품": product,
            "정규화": avg.get(product, 0),
        })

hm_df = pd.DataFrame(heatmap_data)
pivot = hm_df.pivot_table(index="제품", columns="세그먼트", values="정규화")
pivot = pivot.reindex(top10_names)
pivot = pivot[seg_labels]

fig, ax = plt.subplots(figsize=(14, 6))
im = ax.imshow(pivot.values, aspect="auto", cmap="YlOrRd", interpolation="nearest")

ax.set_yticks(range(len(top10_names)))
ax.set_yticklabels(top10_names, fontsize=9)
ax.set_xticks(range(len(seg_labels)))
ax.set_xticklabels(seg_labels, rotation=45, ha="right", fontsize=9)

for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        val = pivot.values[i, j]
        if not np.isnan(val):
            color = "white" if val > 80 else "black"
            ax.text(j, i, f"{val:.0f}", ha="center", va="center", fontsize=8, color=color)

plt.colorbar(im, ax=ax, shrink=0.8, label="정규화 점수")
ax.set_title("세그먼트별 앵커 정규화 TOP 10 히트맵 (VT 리들샷 100 = 100)", fontsize=13, fontweight="bold", pad=12)
plt.tight_layout()

fig.savefig(CHART_DIR / "14_segment_heatmap.png", dpi=150, bbox_inches="tight", facecolor="white")
print("  저장: 14_segment_heatmap.png")
plt.show()

---

## 15. 주요 제품 (VT/본셉 제외)

In [ ]:
other_products = [
    p for p in anchor_ranking.head(20)["제품"].tolist()
    if not p.startswith("VT ") and not p.startswith("본셉")
]
print(f"주요 비-VT/비-본셉 제품: {other_products}")

plot_products(
    base_df, other_products[:7],
    "주요 제품 검색 트렌드 (VT/본셉 제외)",
    filename="15_other_major_products.png",
    annotate_peaks=True,
)

---

## 16. 결승전 결과

In [ ]:
def show_finals_ranking(finals_df, seg_type=None, seg_label=None, title="전체"):
    """결승전 순위 출력"""
    if seg_type:
        sub = finals_df[
            (finals_df["segment_type"] == seg_type) &
            (finals_df["segment_label"] == seg_label)
        ]
    else:
        sub = finals_df[finals_df["segment_type"].isna()]

    if sub.empty:
        print(f"  {title}: 데이터 없음")
        return None

    ranking = (
        sub.groupby("keyword_group")["ratio"]
        .mean()
        .sort_values(ascending=False)
        .reset_index()
    )
    ranking.columns = ["제품", "평균 ratio"]
    ranking.index = range(1, len(ranking) + 1)
    ranking.index.name = "순위"
    ranking["평균 ratio"] = ranking["평균 ratio"].round(2)
    print(f"\n=== {title} 결승전 결과 ===")
    display(ranking)
    return ranking


# 전체 결승전
show_finals_ranking(finals_df, title="전체")

# 성별
for label in ["남성", "여성"]:
    show_finals_ranking(finals_df, "gender", label, title=f"성별 - {label}")

# 연령대
for label in ["10대", "20대", "30대", "40대", "50대", "60대"]:
    show_finals_ranking(finals_df, "age", label, title=f"연령대 - {label}")

# 기기
for label in ["PC", "모바일"]:
    show_finals_ranking(finals_df, "device", label, title=f"기기 - {label}")

---

## 17. 전체 앵커 정규화 순위 (부록)

In [ ]:
print("=== 전체 앵커 정규화 순위 (부록) ===")
display(anchor_ranking)

In [ ]:
print(f"차트 저장 완료: {CHART_DIR}")
print(f"생성된 차트: {len(list(CHART_DIR.glob('*.png')))}개")
for p in sorted(CHART_DIR.glob("*.png")):
    print(f"  {p.name}")